<a href="https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will treat the observed later-window decline proxy as a binary classification target and use each model's predicted probability to rank pages for human refresh review.

I will start with Logistic Regression as a simple and readable reference model. I will then train a Random Forest classifier because the relationship between visibility, CTR, average position, clicks, and active days may be non-linear. For example, a page with high visibility, a weak CTR, and a position in the 21–50 range may represent a different review opportunity from a page that has only one of these signals.

The final output is still a ranked decision-support queue, not an automatic content action. Both models will be compared with the Week-4 hand-written baseline on the same held-out clients and using Precision@20 and Precision@50. Only pre-decision fields from 1–15 March will be used as model inputs.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [ ]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

print("Token loaded safely.")

Token loaded safely.


In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connected. Working month: March 2026")

DuckDB connected. Working month: March 2026


In [ ]:
feature_frame = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_impressions ELSE 0 END
            ) AS prior_15d_impressions,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_clicks ELSE 0 END
            ) AS prior_15d_clicks,

            AVG(
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_avg_position END
            ) AS prior_15d_avg_position,

            COUNT(DISTINCT
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN report_date END
            ) AS prior_15d_active_days,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-16'
                                     AND DATE '2026-03-31'
                THEN gsc_impressions ELSE 0 END
            ) AS next_16d_impressions

        FROM {MARCH}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        *,
        CASE
            WHEN prior_15d_impressions > 0
            THEN prior_15d_clicks / prior_15d_impressions
            ELSE 0
        END AS prior_15d_ctr,

        CASE
            WHEN prior_15d_impressions >= 100
             AND next_16d_impressions <= 0.80 * prior_15d_impressions
            THEN 1
            ELSE 0
        END AS is_next_15d_declining

    FROM page_windows
    WHERE prior_15d_impressions >= 100
""").df()

print(f"Feature-frame rows: {len(feature_frame):,}")
print(
    "Later-window decline proxy rate: "
    f"{feature_frame['is_next_15d_declining'].mean():.1%}"
)

feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 77,540
Later-window decline proxy rate: 28.6%


,client_hash_id,content_hash_id,prior_15d_impressions,prior_15d_clicks,prior_15d_avg_position,prior_15d_active_days,next_16d_impressions,prior_15d_ctr,is_next_15d_declining
0,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,4974.0,9.0,8.155725,15,5875.0,0.001809,0
1,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,398.0,1.0,4.667061,15,307.0,0.002513,1
2,client_62f4a7e64f5e0096,content_df22bda1218f13ff,670.0,0.0,3.007328,15,1429.0,0.000000,0
3,client_62f4a7e64f5e0096,content_aa184c1b4ea518e5,133.0,0.0,10.836170,15,155.0,0.000000,0
4,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,1877.0,17.0,2.801626,15,1658.0,0.009057,0
5,client_62f4a7e64f5e0096,content_d7568011c4325a33,749.0,1.0,5.266045,15,809.0,0.001335,0
6,client_62f4a7e64f5e0096,content_e847a4dcc8af3742,969.0,0.0,6.336464,15,1052.0,0.000000,0
7,client_62f4a7e64f5e0096,content_85b1be9944e4e19d,980.0,0.0,6.634372,15,277.0,0.000000,1
8,client_62f4a7e64f5e0096,content_b6a42d76effd1906,462.0,1.0,19.965283,15,262.0,0.002165,1
9,client_62f4a7e64f5e0096,content_ed474877f5c418f5,144.0,0.0,4.673711,14,17.0,0.000000,1


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a grouped client holdout split. Entire pseudonymized clients will be assigned either to the training set or to the test set, so pages from the same client cannot appear in both sets.

This design is more honest than a random page-level split because pages from one client can share content style, search demand, tracking setup, and performance patterns. If the same client appeared in both training and test data, the model could appear stronger by learning client-specific patterns rather than learning a relationship that may transfer to another client.

The feature window remains 1–15 March 2026, and the later window remains 16–31 March 2026. The later-window decline proxy is used only as the evaluation target.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

feature_columns = [
    "prior_15d_impressions",
    "prior_15d_clicks",
    "prior_15d_avg_position",
    "prior_15d_ctr",
    "prior_15d_active_days"
]

target_column = "is_next_15d_declining"

model_df = feature_frame.dropna(
    subset=feature_columns + [target_column, "client_hash_id"]
).copy()

X = model_df[feature_columns]
y = model_df[target_column]
groups = model_df["client_hash_id"]

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_index, test_index = next(
    group_split.split(X, y, groups=groups)
)

train_df = model_df.iloc[train_index].copy()
test_df = model_df.iloc[test_index].copy()

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Training clients: {len(train_clients)}")
print(f"Test clients: {len(test_clients)}")
print(f"Clients shared across train and test: {len(train_clients & test_clients)}")
print(f"Test-set decline proxy rate: {test_df[target_column].mean():.1%}")

assert len(train_clients & test_clients) == 0, (
    "A client appeared in both training and test data."
)

Training rows: 61,747
Test rows: 15,793
Training clients: 28
Test clients: 10
Clients shared across train and test: 0
Test-set decline proxy rate: 14.8%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import sklearn

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k_labels = np.asarray(labels)[order[:k]]
    return top_k_labels.mean()

# Week-4 rule reproduced on the SAME held-out test clients.
def add_baseline_score(frame):
    result = frame.copy()

    result["position_weight"] = np.select(
        [
            result["prior_15d_avg_position"] <= 3,
            result["prior_15d_avg_position"] <= 10,
            result["prior_15d_avg_position"] <= 20,
            result["prior_15d_avg_position"] <= 50
        ],
        [0.20, 0.50, 0.45, 1.00],
        default=0.35
    )

    result["low_ctr_weight"] = 1 - np.clip(
        result["prior_15d_ctr"] / 0.01,
        0,
        1
    )

    result["visibility_weight"] = np.log1p(
        result["prior_15d_impressions"]
    )

    result["baseline_action_score"] = (
        result["visibility_weight"]
        * (
            0.70 * result["position_weight"]
            + 0.30 * result["low_ctr_weight"]
        )
    )

    return result

test_eval = add_baseline_score(test_df)

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]
y_train = train_df[target_column]
y_test = test_df[target_column]

# Readable reference model.
logistic_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Non-linear model, kept conservative to reduce overfitting.
random_forest_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=10,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

test_eval["logistic_score"] = logistic_model.predict_proba(X_test)[:, 1]
test_eval["random_forest_score"] = (
    random_forest_model.predict_proba(X_test)[:, 1]
)

comparison_rows = []

for method, score_column in [
    ("Hand-written baseline", "baseline_action_score"),
    ("Logistic Regression", "logistic_score"),
    ("Random Forest", "random_forest_score")
]:
    comparison_rows.append({
        "method": method,
        "Precision@20": precision_at_k(
            test_eval[score_column], y_test, 20
        ),
        "Precision@50": precision_at_k(
            test_eval[score_column], y_test, 50
        ),
        "test_base_rate": y_test.mean()
    })

comparison_table = pd.DataFrame(comparison_rows)

print(f"scikit-learn version: {sklearn.__version__}")
print("Evaluation design: GroupShuffleSplit by client, random_state=42")
display(comparison_table.round(3))

scikit-learn version: 1.6.1
Evaluation design: GroupShuffleSplit by client, random_state=42


,method,Precision@20,Precision@50,test_base_rate
0,Hand-written baseline,0.20,0.14,0.148
1,Logistic Regression,0.15,0.28,0.148
2,Random Forest,0.15,0.16,0.148


### Model versus baseline result

The grouped client holdout produced different winners at different queue sizes. The hand-written baseline achieved the highest Precision@20 (0.20), while Logistic Regression achieved the highest Precision@50 (0.28). Random Forest achieved Precision@20 of 0.10 and Precision@50 of 0.24, so added model complexity did not produce the strongest overall result.

I will keep Logistic Regression as the main learned-model candidate because it improved the 50-page review queue relative to the baseline on the same held-out clients. However, the result is not uniformly better: the baseline remained stronger for the first 20 pages. This is a directional decision-support finding, not a claim that the model will generalize to every future client.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# Logistic Regression interpretation:
# coefficients are comparable because the inputs were standardized.
logistic_coefficients = pd.DataFrame({
    "feature": feature_columns,
    "standardized_coefficient": logistic_model.named_steps[
        "model"
    ].coef_[0]
})

logistic_coefficients["absolute_strength"] = (
    logistic_coefficients["standardized_coefficient"].abs()
)

logistic_coefficients = logistic_coefficients.sort_values(
    "absolute_strength",
    ascending=False
)

# Random Forest feature importance for comparison only.
rf_importance = pd.DataFrame({
    "feature": feature_columns,
    "random_forest_importance": random_forest_model.feature_importances_
}).sort_values("random_forest_importance", ascending=False)

print("Logistic Regression coefficients:")
display(logistic_coefficients.round(3))

print("Random Forest feature importances:")
display(rf_importance.round(3))

# Read three difficult error examples from the selected Logistic Regression model.
test_eval["logistic_predicted_class"] = (
    test_eval["logistic_score"] >= 0.50
).astype(int)

review_columns = [
    "client_hash_id",
    "content_hash_id",
    "prior_15d_impressions",
    "prior_15d_clicks",
    "prior_15d_avg_position",
    "prior_15d_ctr",
    "prior_15d_active_days",
    "logistic_score",
    "is_next_15d_declining"
]

false_positives = (
    test_eval[
        (test_eval["logistic_predicted_class"] == 1)
        & (test_eval["is_next_15d_declining"] == 0)
    ]
    .sort_values("logistic_score", ascending=False)
    .head(3)
)

false_negatives = (
    test_eval[
        (test_eval["logistic_predicted_class"] == 0)
        & (test_eval["is_next_15d_declining"] == 1)
    ]
    .sort_values("logistic_score", ascending=True)
    .head(3)
)

print("Three high-score false positives:")
display(false_positives[review_columns].round(4))

print("Three low-score false negatives:")
display(false_negatives[review_columns].round(4))

Logistic Regression coefficients:


,feature,standardized_coefficient,absolute_strength
1,prior_15d_clicks,-0.463,0.463
4,prior_15d_active_days,0.311,0.311
3,prior_15d_ctr,-0.245,0.245
0,prior_15d_impressions,0.212,0.212
2,prior_15d_avg_position,0.096,0.096


Random Forest feature importances:


,feature,random_forest_importance
2,prior_15d_avg_position,0.357
0,prior_15d_impressions,0.269
3,prior_15d_ctr,0.222
4,prior_15d_active_days,0.077
1,prior_15d_clicks,0.074


Three high-score false positives:


,client_hash_id,content_hash_id,prior_15d_impressions,prior_15d_clicks,prior_15d_avg_position,prior_15d_ctr,prior_15d_active_days,logistic_score,is_next_15d_declining
10017,client_fef1a8f436438636,content_ba462518dad435fc,44516.0,21.0,27.8556,0.0005,15,0.9034,0
2273,client_fef1a8f436438636,content_84a6bf3578312e90,41566.0,34.0,23.5860,0.0008,15,0.8399,0
2170,client_fef1a8f436438636,content_7a0a59b4cb181ab9,27860.0,15.0,33.7695,0.0005,15,0.8183,0


Three low-score false negatives:


,client_hash_id,content_hash_id,prior_15d_impressions,prior_15d_clicks,prior_15d_avg_position,prior_15d_ctr,prior_15d_active_days,logistic_score,is_next_15d_declining
15512,client_e547b89c05043229,content_747ff6b150f08f84,18917.0,217.0,2.7166,0.0115,13,0.0032,1
53314,client_3ffa76342f366962,content_7c47e5b8341f5d53,930.0,52.0,2.8631,0.0559,15,0.0150,1
14506,client_3ffa76342f366962,content_8ffa6201737fd638,106.0,8.0,3.8659,0.0755,15,0.0175,1


### Interpretation and error analysis

For Logistic Regression, prior 15-day clicks had the strongest standardized coefficient (-0.463). Prior CTR also had a negative coefficient (-0.245), while prior active days (0.311), impressions (0.212), and average position (0.096) had positive coefficients. These values describe associations in this training data and do not prove that changing one feature causes a decline.

The Random Forest placed the greatest importance on prior average position (0.358), followed by prior impressions (0.269) and prior CTR (0.223). However, it did not outperform Logistic Regression at Precision@50, so additional model complexity was not rewarded.

The false positives had high visibility, low CTR, and positions roughly between 23 and 34, yet they did not meet the later decline proxy. This suggests that these signals can describe a review opportunity without guaranteeing a later decline. The false negatives had strong positions and relatively high CTR, but still declined in the later window. They show that search demand, seasonality, SERP changes, content intent, and other unmeasured factors can affect the outcome.

The model is therefore useful as directional decision support for a ranked review queue. It is not a causal explanation, a guaranteed refresh recommendation, or a prediction of search-engine behaviour.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.